# Neologism sweep: do new words end up where their meaning says they should?
One new tied row of `E` per word (Phase A by default: only that row trains), trained on TinyStories sentences where
a phrase was replaced by `{w}`. Every word gets the same tests; then words are compared **by type**.

| question | metric | good looks like |
|---|---|---|
| is it used where the phrase was? | `p_eval`: p(NEW) at held-out phrase slots | high |
| only there? | `p_ctrl`: p(NEW) at random slots | ~0 |
| does it **mean** the right thing? | `lift_comp` = p(NEW) where a same-meaning word occurs (" cake" for *ice cream*) / `p_ctrl` | >> 1 |
|  |  (each competitor word weighs equally; words with < MIN_SLOT_ROWS rows are skipped; NaN if none) | |
| or does it just **echo** a part of the phrase? | `lift_near` = p(NEW) where a phrase part occurs alone (" ice" without "cream") / `p_ctrl` | < `lift_comp` for idioms / compounds |
| does the rest of the sentence read as if the phrase were there? | `carried` = 1 - KL(phrase‖NEW) / KL(phrase‖phrase deleted), next 5 tokens | 1 = all the phrase's info, 0 = no better than deleting it |
| where is the row? | centered-cos neighbours (active rows), `comp@10`, `cos_comp`, `cos_near`, `cos_gold` | competitors among neighbours |
| does the model read NEW like the phrase? | `cos_gap_h` / `cos_gap_mid`: centered cos(h at NEW, h at phrase end), matched - shuffled | > 0 |
| where does it fire wrongly? | top control slots by p(NEW), with their context | should look like phrase contexts |

Data: `data/words/<word>/` from `scripts/build_word_data.py` (116 words: the original 6 + `scripts/word_list.py`).
Results: `outputs/sweep/<tag>/<word>.json` (one file per word; finished words are skipped on rerun) + `summary.csv`.

In [ ]:
# ---- 0. paths + config ----
import os, sys, json, time, random, collections
from pathlib import Path

# code root = the repo root (has model.py); data/words and outputs/ are relative to it (same as `python -m experiments.neologism`)
ROOT = next(q for p in [Path.cwd(), *Path.cwd().parents] for q in (p, p / "src/model_new_words") if (q / "model.py").exists() and (q / "experiments").is_dir())
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 70)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})

CFG = dict(
    MODEL="roneneldan/TinyStories-33M",
    DEVICE="cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"),
    DTYPE="float32",
    WORDS="all",                  # "all" or a list of word dirs, e.g. ["syn_huge", "cmp_fire_truck"]
    STATUS=("keep", "maybe"),     # word_list.py status; the original 6 words count as "keep"
    TYPES=None,                   # or e.g. ["idiom", "cmp"]
    INIT="mean_h",                # mean_emb | mean_h | in_span | gold (falls back to mean_h if the word has no gold)
    N_TRAIN=200,
    LR_NEW=5e-3,
    LR_REST=0.0,                  # 0 = Phase A (only the new row); > 0 = Phase B (base model restored per word)
    STEPS=300, BS=16, LOG_EVERY=50,
    N_EVAL=200, N_PER_SLOT=100, N_CTRL=300, N_EQUIV=64,
    MIN_SLOT_ROWS=20,             # competitor / near-miss words with fewer validation rows are ignored (too noisy)
    OUT="outputs/sweep",
    SEED=0,
    RERUN=False,                  # True = recompute words that already have a result file
)
TAG = f"{CFG['INIT']}_lrrest{CFG['LR_REST']:g}_n{CFG['N_TRAIN']}_st{CFG['STEPS']}_s{CFG['SEED']}"
OUTDIR = Path(CFG["OUT"]) / TAG; OUTDIR.mkdir(parents=True, exist_ok=True)
TYPE_ORDER = ["syn", "comp", "idiom", "cmp", "act", "tmp", "spat", "blend", "abs"]
RUNG2TYPE = {"synonym": "syn", "phrase_compositional": "comp", "phrase_idiom": "idiom", "phrase_lexicalized": "cmp",
             "phrase_action": "act", "phrase_temporal": "tmp", "phrase_spatial": "spat", "blend": "blend", "phrase_abstract": "abs"}
print(ROOT, "|", CFG["DEVICE"], "|", OUTDIR)

In [ ]:
# ---- 1. repo code ----
from model import Model
from train import TrainConfig, train
from dataset import list_words, load_word, word_meta
from experiments.neologism import make_inits, slot_metrics, equivalence, geometry

In [ ]:
# ---- 2. which words ----
def wtype(meta): return meta.get("type") or RUNG2TYPE[meta["rung"]]

rows = []
for w in list_words():
    mt = word_meta(w)
    rows.append(dict(word=w, type=wtype(mt), status=mt.get("status", "keep"), train=mt["counts"]["train_pool"],
                     eval=mt["counts"]["eval"], comp=sum(mt["counts"]["competitor"].values()),
                     near=sum(mt["counts"]["nearmiss"].values()), phrases=" / ".join(mt["phrases"][:3]),
                     competitors=", ".join(mt["competitors"]), nearmiss=", ".join(mt["nearmiss"]), gold=mt.get("gold")))
WL = pd.DataFrame(rows)
WL["type"] = pd.Categorical(WL["type"], TYPE_ORDER, ordered=True)
WL = WL.sort_values(["type", "word"]).reset_index(drop=True)
sel = WL[WL.status.isin(CFG["STATUS"])]
if CFG["WORDS"] != "all": sel = sel[sel.word.isin(CFG["WORDS"])]
if CFG["TYPES"]: sel = sel[sel.type.isin(CFG["TYPES"])]
WORDS = sel.word.tolist()
print(f"{len(WL)} words built, {len(WORDS)} selected:", sel.type.value_counts().reindex(TYPE_ORDER).to_dict())
sel[["word", "type", "status", "train", "eval", "comp", "near", "phrases", "competitors", "nearmiss", "gold"]]

In [ ]:
# ---- 3. model (loaded once; one extra row reused for every word) ----
torch.manual_seed(CFG["SEED"])
m = Model.load(CFG["MODEL"], device=CFG["DEVICE"], dtype=CFG["DTYPE"])
new_id = m.add_row()
E0 = m.E.detach().clone()
others0 = ({n: p.detach().clone() for n, p in m.hf.named_parameters() if p is not m.E} if CFG["LR_REST"] > 0 else {})
MID = len(m.blocks) // 2
print("E", tuple(m.E.shape), "new_id", new_id, "blocks", len(m.blocks))

def restore():
    with torch.no_grad():
        m.E.copy_(E0)
        for n, p in m.hf.named_parameters():
            if n in others0: p.copy_(others0[n])

In [ ]:
# ---- 4. metrics ----
def one_tok(w):
    ids = m.tok.encode(" " + w.strip(), add_special_tokens=False)
    return ids[0] if len(ids) == 1 else None

def per_slot(rows, n, seed):
    """Up to n rows per slot word (competitor/nearmiss files are grouped by slot word, so [:N] would take one word)."""
    g = collections.defaultdict(list)
    for r in rows: g[r["slot"]].append(r)
    out = []
    for s, rs in g.items():
        random.Random(seed).shuffle(rs); out += rs[:n]
    return out

@torch.no_grad()
def kl_drop(rows, n_next=5, bs=32):
    """KL(p_orig || p_drop): same as equivalence(), but the phrase is DELETED instead of replaced by NEW.
    The denominator of `carried`: how much the phrase matters for the next tokens at all."""
    out = []
    for i in range(0, len(rows), bs):
        pairs = [p for p in (m.encode_pair(r, new_id) for r in rows[i:i + bs]) if p[3] > 0]
        if not pairs: continue
        drop = [p[0][:p[2]] + p[0][p[2] + 1:] for p in pairs]                 # left + right
        a, aa = m.pad(drop); b, ba = m.pad([p[1] for p in pairs])
        la = m.forward(a, aa)[..., :new_id].float().log_softmax(-1)
        lb = m.forward(b, ba)[..., :new_id].float().log_softmax(-1)
        for j, (ids_new, ids_orig, slot, nr) in enumerate(pairs):
            k = min(n_next, nr); pb = len(ids_orig) - nr - 1
            A, B = la[j, slot - 1:slot - 1 + k], lb[j, pb:pb + k]
            out.append(F.kl_div(A, B, log_target=True, reduction="none").sum(-1).mean().item())
    return torch.tensor(out)

@torch.no_grad()
def slot_rep_gap(rows):
    """Centered cos between the state AT NEW and the state at the phrase's last token, matched minus shuffled.
    h = pre-decode vector, mid = residual after the middle block."""
    pairs = [m.encode_pair(r, new_id) for r in rows]
    a = m.states_at_last([p[0][:p[2] + 1] for p in pairs], layers=[MID])
    b = m.states_at_last([p[1][:len(p[1]) - p[3]] for p in pairs], layers=[MID])
    out = {}
    for name, x, y in [("h", a["h"], b["h"]), ("mid", a["resid"][MID], b["resid"][MID])]:
        mu = y.mean(0); x, y = x - mu, y - mu
        out[f"cos_gap_{name}"] = (F.cosine_similarity(x, y).mean() - F.cosine_similarity(x, y.roll(1, 0)).mean()).item()
    return out

@torch.no_grad()
def evaluate(W, e, kl_d):
    """All metrics for row value e (already set in E[new_id]). W = per-word context from setup_word."""
    Ecur = m.E.detach()[:new_id].float()
    H = W["H"] if CFG["LR_REST"] == 0 else {s: m.slot_states(r)["h"] for s, r in W["sets"].items()}
    r = {}
    for s in ["eval", "control"]:
        sm = slot_metrics(e, H[s], Ecur)
        r[f"p_{s}"], r[f"top1_{s}"] = sm["p_new"], sm["top1_frac"]
    for s in ["competitor", "nearmiss"]:
        by = collections.defaultdict(list)
        for i, row in enumerate(W["sets"][s]): by[row["slot"]].append(i)
        by = {w: idx for w, idx in by.items() if len(idx) >= CFG["MIN_SLOT_ROWS"]}
        ps = {w: slot_metrics(e, H[s][idx], Ecur)["p_new"] for w, idx in by.items()}
        r[f"p_{s}_by"] = ps
        r[f"p_{s}"] = float(np.mean(list(ps.values()))) if ps else float("nan")   # each slot word weighs equally
    r["lift_comp"] = r["p_competitor"] / max(r["p_control"], 1e-12)
    r["lift_near"] = r["p_nearmiss"] / max(r["p_control"], 1e-12)
    q = equivalence(m, W["equiv"], new_id)
    r["kl_new"], r["kl_drop"] = q["kl"].mean().item(), kl_d.mean().item()
    r["carried"] = 1 - r["kl_new"] / max(r["kl_drop"], 1e-9)
    r["nll_new"], r["nll_orig"] = q["nll_new"].mean().item(), q["nll_orig"].mean().item()
    r.update(slot_rep_gap(W["equiv"]))
    g = geometry(m, e, new_id, W["act"], W["mu"], W["refs"], k=10)
    r["neighbors"] = [w for w, _ in g["neighbors"]]
    r["norm_ratio"] = g["norm"] / W["row_norm"]
    cos = g["cos"]
    r["cos_comp"] = float(np.mean([cos[w] for w in W["comp_words"]])) if W["comp_words"] else float("nan")
    r["cos_near"] = float(np.mean([cos[w] for w in W["near_words"]])) if W["near_words"] else float("nan")
    r["cos_gold"] = cos.get(W["gold"], float("nan")) if W["gold"] else float("nan")
    nb = [w.strip().lower() for w in r["neighbors"]]
    r["comp@10"] = float(np.mean([w in nb for w in W["comp_words"]])) if W["comp_words"] else float("nan")
    r["near@10"] = float(np.mean([w in nb for w in W["near_words"]])) if W["near_words"] else float("nan")
    # where NEW fires most outside its phrase: top control slots
    pc = (H["control"] @ e.float() - torch.logsumexp(torch.cat([H["control"] @ Ecur.T, (H["control"] @ e.float())[:, None]], 1), 1)).exp()
    top = pc.topk(min(3, len(pc))).indices.tolist()
    r["ctrl_top"] = [(round(pc[i].item(), 3), "…" + W["sets"]["control"][i]["left"][-60:].replace("\n", " ") + " ▮") for i in top]
    return r

def setup_word(word):
    meta = word_meta(word)
    train_rows = load_word(word, "train_pool", n=CFG["N_TRAIN"])
    sets = dict(eval=load_word(word, "eval")[:CFG["N_EVAL"]],
                competitor=per_slot(load_word(word, "competitor"), CFG["N_PER_SLOT"], CFG["SEED"]),
                nearmiss=per_slot(load_word(word, "nearmiss"), CFG["N_PER_SLOT"], CFG["SEED"]),
                control=load_word(word, "control")[:CFG["N_CTRL"]])
    act = m.active_ids(train_rows + sets["eval"] + sets["control"])
    gold = meta["gold"].strip() if meta.get("gold") else None
    gold_id = one_tok(gold) if gold else None
    inits, info = make_inits(m, train_rows, new_id, act, gold_id=gold_id)
    comp_words = [w for w in meta["competitors"] if one_tok(w) is not None]
    near_words = [w for w in meta["nearmiss"] if one_tok(w) is not None]
    refs = {w: one_tok(w) for w in dict.fromkeys(([gold] if gold_id else []) + comp_words + near_words)}
    H = {s: m.slot_states(r)["h"] for s, r in sets.items()}
    return dict(meta=meta, train=train_rows, sets=sets, act=act, inits=inits, mu=info["mu"], row_norm=info["row_norm"].item(),
                refs=refs, comp_words=comp_words, near_words=near_words, gold=gold if gold_id else None, H=H,
                equiv=[r for r in sets["eval"] if r["right"].strip()][:CFG["N_EQUIV"]])

def run_word(word):
    restore()
    W = setup_word(word)
    init = W["inits"].get(CFG["INIT"], W["inits"]["mean_h"])
    m.set_row(new_id, init)
    kl_d = kl_drop(W["equiv"])                       # base model, independent of the new row in Phase A
    before = evaluate(W, init, kl_d)

    def logger(step, mm, e, loss):
        H = W["H"] if CFG["LR_REST"] == 0 else {s: mm.slot_states(W["sets"][s])["h"] for s in ["eval", "control"]}
        Ecur = mm.E.detach()[:new_id].float()
        return {s: slot_metrics(e, H[s], Ecur)["p_new"] for s in ["eval", "control"]}

    tc = TrainConfig(steps=CFG["STEPS"], bs=CFG["BS"], lr_new=CFG["LR_NEW"], lr_rest=CFG["LR_REST"],
                     log_every=CFG["LOG_EVERY"], seed=CFG["SEED"])
    t0 = time.time()
    traj, logs = train(m, W["train"], new_id, init, tc, logger=logger)
    after = evaluate(W, m.E.detach()[new_id].float(), kl_drop(W["equiv"]) if CFG["LR_REST"] > 0 else kl_d)
    res = dict(word=word, type=wtype(W["meta"]), status=W["meta"].get("status", "keep"), phrases=W["meta"]["phrases"],
               competitors=W["comp_words"], nearmiss=W["near_words"], gold=W["gold"], cfg=CFG, train_s=time.time() - t0,
               before=before, after=after, curve=[dict(step=l["step"], loss=l["loss"], p_eval=l["eval"], p_ctrl=l["control"]) for l in logs])
    (OUTDIR / f"{word}.json").write_text(json.dumps(res, indent=1, default=float))
    torch.save(traj, OUTDIR / f"{word}.traj.pt")
    return res

In [ ]:
# ---- 5. run (resumable: words with a result file are skipped unless RERUN) ----
results = {}
for i, w in enumerate(WORDS):
    f = OUTDIR / f"{w}.json"
    if f.exists() and not CFG["RERUN"]:
        results[w] = json.loads(f.read_text()); continue
    t0 = time.time()
    r = results[w] = run_word(w)
    a, b = r["after"], r["before"]
    print(f"[{i + 1:3d}/{len(WORDS)}] {w:<30} p_eval {b['p_eval']:.2f}->{a['p_eval']:.2f}  lift_comp {a['lift_comp']:7.1f}  "
          f"lift_near {a['lift_near']:7.1f}  carried {a['carried']:+.2f}  nbrs {' '.join(x.strip() for x in a['neighbors'][:5])}  "
          f"({time.time() - t0:.0f}s)")

In [ ]:
# ---- 6. summary table ----
COLS = ["p_eval", "p_control", "lift_comp", "lift_near", "carried", "kl_new", "kl_drop", "cos_gap_h", "cos_gap_mid",
        "comp@10", "near@10", "cos_comp", "cos_near", "cos_gold", "norm_ratio"]
R = pd.DataFrame([dict(word=w, type=r["type"], status=r["status"], **{c: r["after"][c] for c in COLS},
                       **{c + "_0": r["before"][c] for c in ["p_eval", "lift_comp", "carried", "cos_gap_h"]},
                       neighbors=" ".join(x.strip() for x in r["after"]["neighbors"][:6]))
                  for w, r in results.items()])
R["type"] = pd.Categorical(R["type"], TYPE_ORDER, ordered=True)
R = R.sort_values(["type", "word"]).reset_index(drop=True)
R.to_csv(OUTDIR / "summary.csv", index=False)
TS = [t for t in TYPE_ORDER if t in set(R.type)]
print("per-type medians")
display(R.groupby("type", observed=True)[["p_eval", "p_control", "lift_comp", "lift_near", "carried", "cos_gap_h", "comp@10", "cos_comp", "cos_near"]]
        .median().round(3))
R[["word", "type", "status", "p_eval", "p_control", "lift_comp", "lift_near", "carried", "cos_gap_h", "comp@10", "cos_comp", "cos_near", "neighbors"]].round(3)

In [ ]:
# ---- 7. by type: one dot per word, bar = median ----
def by_type(ax, col, log=False, ref=None, title=None):
    for i, t in enumerate(TS):
        v = R[R.type == t][col].astype(float).values
        ax.scatter(i + np.random.default_rng(0).uniform(-.18, .18, len(v)), v, s=20, alpha=.8, color=f"C{i}")
        ax.hlines(np.nanmedian(v), i - .3, i + .3, color="k")
    ax.set_xticks(range(len(TS))); ax.set_xticklabels(TS); ax.set_title(title or col)
    if log: ax.set_yscale("log")
    if ref is not None: ax.axhline(ref, ls="--", c="gray", lw=1)

fig, ax = plt.subplots(2, 3, figsize=(17, 7.5))
by_type(ax[0, 0], "p_eval", title="p(NEW) at held-out phrase slots")
by_type(ax[0, 1], "lift_comp", log=True, ref=1, title="lift where a SAME-MEANING word occurs (meaning)")
by_type(ax[0, 2], "lift_near", log=True, ref=1, title="lift where a phrase PART occurs alone (echo)")
by_type(ax[1, 0], "carried", ref=0, title="carried: 1 - KL_new / KL_deleted (next 5 tokens)")
by_type(ax[1, 1], "cos_gap_h", ref=0, title="state at NEW vs phrase end: matched - shuffled")
by_type(ax[1, 2], "cos_comp", ref=0, title="centered cos(e_new, same-meaning rows)")
plt.tight_layout(); plt.show()

In [ ]:
# ---- 8. meaning vs echo, one point per word ----
fig, ax = plt.subplots(1, 2, figsize=(15, 6.2))
for i, t in enumerate(TS):
    s = R[R.type == t]
    ax[0].scatter(s.lift_near, s.lift_comp, s=26, color=f"C{i}", label=t)
    ax[1].scatter(s.cos_near, s.cos_comp, s=26, color=f"C{i}", label=t)
    for _, r in s.iterrows():
        lab = r.word.split("_", 1)[1]
        ax[0].annotate(lab, (r.lift_near, r.lift_comp), fontsize=6.5, alpha=.8)
        ax[1].annotate(lab, (r.cos_near, r.cos_comp), fontsize=6.5, alpha=.8)
v = R[["lift_near", "lift_comp"]].astype(float).replace(0, np.nan)
lo, hi = np.nanmin(v.values) / 2, np.nanmax(v.values) * 2
ax[0].plot([lo, hi], [lo, hi], "--", c="gray"); ax[0].set(xscale="log", yscale="log", xlabel="lift_near (echo)", ylabel="lift_comp (meaning)",
                                                           title="usage: above the diagonal = meaning beats echo")
ax[1].plot([-.5, 1], [-.5, 1], "--", c="gray"); ax[1].set(xlabel="cos to phrase parts", ylabel="cos to same-meaning words",
                                                          title="geometry: above the diagonal = row sits with its meaning")
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
# ---- 9. before (init) vs after training ----
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for a, c in zip(ax, ["p_eval", "lift_comp", "carried", "cos_gap_h"]):
    for i, t in enumerate(TS):
        s = R[R.type == t]; a.scatter(s[c + "_0"], s[c], s=18, color=f"C{i}", label=t)
    x = R[[c, c + "_0"]].astype(float).values; lo, hi = np.nanmin(x), np.nanmax(x)
    a.plot([lo, hi], [lo, hi], "--", c="gray", lw=1); a.set(title=c, xlabel="init", ylabel="trained")
    if c == "lift_comp": a.set(xscale="log", yscale="log")
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

In [ ]:
# ---- 10. training curves by type ----
fig, ax = plt.subplots(2, len(TS), figsize=(3.1 * len(TS), 5.5), sharex=True, squeeze=False)
for j, t in enumerate(TS):
    for w in R[R.type == t].word:
        c = results[w]["curve"]
        ax[0, j].plot([x["step"] for x in c], [x["p_eval"] for x in c], lw=1)
        ax[1, j].plot([x["step"] for x in c], [x["p_ctrl"] for x in c], lw=1)
    ax[0, j].set_title(t); ax[1, j].set_yscale("log"); ax[1, j].set_xlabel("step")
ax[0, 0].set_ylabel("p(NEW) eval slots"); ax[1, 0].set_ylabel("p(NEW) control slots")
plt.tight_layout(); plt.show()

In [ ]:
# ---- 11. one word up close (change WORD) ----
WORD = WORDS[0]
r = results[WORD]; a, b = r["after"], r["before"]
print(f"{WORD} ({r['type']}, {r['status']})  phrases {r['phrases']}  gold {r['gold']}")
print("neighbours  init :", " ".join(x.strip() for x in b["neighbors"]))
print("neighbours  after:", " ".join(x.strip() for x in a["neighbors"]))
print("p(NEW) by competitor slot:", {k: round(v, 4) for k, v in a["p_competitor_by"].items()}, " control", round(a["p_control"], 5))
print("p(NEW) by near-miss slot :", {k: round(v, 4) for k, v in a["p_nearmiss_by"].items()})
print("where it fires most on control slots:")
for p, ctx in a["ctrl_top"]: print(f"   {p:.3f}  {ctx}")
print({k: round(a[k], 3) for k in ["p_eval", "carried", "kl_new", "kl_drop", "cos_gap_h", "cos_gap_mid", "cos_comp", "cos_near", "norm_ratio"]})